In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import json
import re
import math
import statistics

# Load data sau khi clean

In [2]:
OUT_DIR = Path("../seed_exports")
CLEAN_JSONL = OUT_DIR / "wiki_mcq_seed_clean.jsonl"

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

# Chỉ chạy nếu notebook không còn biến cleaned
cleaned = read_jsonl(CLEAN_JSONL)

print("Loaded clean rows:", len(cleaned))

Loaded clean rows: 7946


### Parse question và choices

In [3]:
ANSWER_LETTERS = ["A", "B", "C", "D"]

def clean_text(x):
    if x is None:
        return ""
    x = str(x)
    x = x.replace("\u00a0", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x


def parse_user_content(user_content):
    """
    Parse format:
    Câu hỏi: ...
    A. ...
    B. ...
    C. ...
    D. ...

    Return:
    {
      "question": str,
      "choices": {"A": ..., "B": ..., "C": ..., "D": ...}
    }
    """
    lines = user_content.splitlines()

    question_lines = []
    choices = {}
    current_choice = None

    for line in lines:
        line = line.rstrip()

        m = re.match(r"^([ABCD])\.\s*(.*)$", line.strip())
        if m:
            current_choice = m.group(1)
            choices[current_choice] = m.group(2).strip()
            continue

        if current_choice is not None:
            # Nếu option bị wrap nhiều dòng
            choices[current_choice] += " " + line.strip()
        else:
            question_lines.append(line.strip())

    question = "\n".join(question_lines)
    question = re.sub(r"^Câu hỏi:\s*", "", question).strip()

    choices = {k: clean_text(v) for k, v in choices.items()}

    if set(choices.keys()) != set(ANSWER_LETTERS):
        return None

    return {
        "question": clean_text(question),
        "choices": choices,
    }


def get_answer(item):
    ans = item.get("metadata", {}).get("answer")
    if ans in ANSWER_LETTERS:
        return ans

    content = item["messages"][1]["content"]
    m = re.match(r"^Đáp án:\s*([ABCD])$", content.strip())
    return m.group(1) if m else None

### Word length helpers

In [4]:
def vietnamese_word_count(text):
    """
    Simple tokenizer theo whitespace / punctuation    
    """
    text = clean_text(text.lower())
    tokens = re.findall(r"[0-9a-zA-ZÀ-ỹ]", text) # tìm tất cả các ký tự
    
    return len(tokens)

def char_len(text):
    return len(clean_text(text))    

### Dataframe answer-length

In [5]:
length_rows = []

parse_errors = []

for i, item in enumerate(cleaned):
    parsed = parse_user_content(item["messages"][0]["content"])
    answer = get_answer(item)

    if parsed is None or answer not in ANSWER_LETTERS:
        parse_errors.append(i)
        continue

    choices = parsed["choices"]
    correct_text = choices[answer]
    distractor_labels = [x for x in ANSWER_LETTERS if x != answer]
    distractor_texts = [choices[x] for x in distractor_labels]

    choice_word_lens = {
        label: vietnamese_word_count(text)
        for label, text in choices.items()
    }

    choice_char_lens = {
        label: char_len(text)
        for label, text in choices.items()
    }

    correct_word_len = choice_word_lens[answer]
    distractor_word_lens = [
        choice_word_lens[label]
        for label in distractor_labels
    ]

    correct_char_len = choice_char_lens[answer]
    distractor_char_lens = [
        choice_char_lens[label]
        for label in distractor_labels
    ]

    longest_word_label = max(choice_word_lens, key=choice_word_lens.get)
    longest_char_label = max(choice_char_lens, key=choice_char_lens.get)

    length_rows.append({
        "row_idx": i,
        "source_id": item["metadata"].get("source_id"),
        "subject": item["metadata"].get("subject"),
        "answer": answer,

        "correct_word_len": correct_word_len,
        "avg_distractor_word_len": sum(distractor_word_lens) / 3,
        "max_distractor_word_len": max(distractor_word_lens),
        "correct_minus_avg_distractor_word_len": correct_word_len - (sum(distractor_word_lens) / 3),

        "correct_char_len": correct_char_len,
        "avg_distractor_char_len": sum(distractor_char_lens) / 3,
        "correct_minus_avg_distractor_char_len": correct_char_len - (sum(distractor_char_lens) / 3),

        "longest_word_label": longest_word_label,
        "longest_word_is_correct": longest_word_label == answer,

        "longest_char_label": longest_char_label,
        "longest_char_is_correct": longest_char_label == answer,

        "question": parsed["question"],
        "correct_text": correct_text,
        "choices": choices,
    })

df_len = pd.DataFrame(length_rows)

print("Parsed rows:", len(df_len))
print("Parse errors:", len(parse_errors))
display(df_len.head())

Parsed rows: 7946
Parse errors: 0


,row_idx,source_id,subject,answer,correct_word_len,avg_distractor_word_len,max_distractor_word_len,correct_minus_avg_distractor_word_len,correct_char_len,avg_distractor_char_len,correct_minus_avg_distractor_char_len,longest_word_label,longest_word_is_correct,longest_char_label,longest_char_is_correct,question,correct_text,choices
0,0,sociology/test/49,sociology,C,56,60.666667,82,-4.666667,75,77.000000,-2.000000,D,False,D,False,Nhóm Quốc gia Islam kêu gọi đến:,Những người Mỹ gốc Phi cảm thấy bị loại trừ kh...,{'A': 'Những người nhập cư thế hệ thứ hai sinh...
1,1,miscellaneous/test/535,miscellaneous,A,8,9.333333,13,-1.333333,9,11.000000,-2.000000,D,False,D,False,Phần nào của phổ điện từ có bước sóng ngắn nhất?,Tia gamma,"{'A': 'Tia gamma', 'B': 'Tia X', 'C': 'Sóng vô..."
2,2,human_aging/test/173,human_aging,C,25,37.333333,44,-12.333333,32,49.000000,-17.000000,A,False,A,False,Các ông bà có cháu đang theo học đại học cho b...,Đi thăm ngẫu nhiên và hỏi ý kiến,{'A': 'Hỏi về sự hài lòng với cuộc sống của họ...
3,3,astronomy/test/11,astronomy,C,98,48.000000,66,50.000000,128,60.666667,67.333333,C,True,C,True,Làm thế nào Eratosthenes ước tính kích thước c...,Bằng cách so sánh độ cao tối đa của Mặt Trời t...,{'A': 'Bằng cách quan sát thời lượng của một s...
4,4,high_school_mathematics/test/166,high_school_mathematics,B,2,1.666667,2,0.333333,2,1.666667,0.333333,A,False,A,False,Tìm $(2^{20} + 2^{20} + 2^{20} +2^{21}) \div 2...,40,"{'A': '20', 'B': '40', 'C': '0', 'D': '10'}"


In [6]:
summary_len = {
    "rows": int(len(df_len)),

    "avg_correct_word_len": round(float(df_len["correct_word_len"].mean()), 3),
    "avg_distractor_word_len": round(float(df_len["avg_distractor_word_len"].mean()), 3),
    "avg_word_len_gap_correct_minus_distractor": round(float(df_len["correct_minus_avg_distractor_word_len"].mean()), 3),

    "avg_correct_char_len": round(float(df_len["correct_char_len"].mean()), 3),
    "avg_distractor_char_len": round(float(df_len["avg_distractor_char_len"].mean()), 3),
    "avg_char_len_gap_correct_minus_distractor": round(float(df_len["correct_minus_avg_distractor_char_len"].mean()), 3),

    "longest_word_choice_is_correct_rate": round(float(df_len["longest_word_is_correct"].mean() * 100), 2),
    "longest_char_choice_is_correct_rate": round(float(df_len["longest_char_is_correct"].mean() * 100), 2),
}

summary_len

{'rows': 7946,
 'avg_correct_word_len': 27.752,
 'avg_distractor_word_len': 26.959,
 'avg_word_len_gap_correct_minus_distractor': 0.793,
 'avg_correct_char_len': 35.596,
 'avg_distractor_char_len': 34.507,
 'avg_char_len_gap_correct_minus_distractor': 1.089,
 'longest_word_choice_is_correct_rate': 27.38,
 'longest_char_choice_is_correct_rate': 27.57}

In [7]:
df_len_by_subject = (
    df_len
    .groupby("subject")
    .agg(
        rows=("row_idx", "count"),
        avg_correct_word_len=("correct_word_len", "mean"),
        avg_distractor_word_len=("avg_distractor_word_len", "mean"),
        avg_word_gap=("correct_minus_avg_distractor_word_len", "mean"),
        longest_word_correct_rate=("longest_word_is_correct", "mean"),
    )
    .reset_index()
)

df_len_by_subject["longest_word_correct_rate"] = (
    df_len_by_subject["longest_word_correct_rate"] * 100
)

df_len_by_subject = df_len_by_subject.sort_values(
    ["longest_word_correct_rate", "rows"],
    ascending=[False, False]
)

display(df_len_by_subject)

,subject,rows,avg_correct_word_len,avg_distractor_word_len,avg_word_gap,longest_word_correct_rate
21,management,119,20.857143,18.756303,2.100840,35.294118
19,logical_fallacies,185,37.627027,34.003604,3.623423,34.594595
25,nutrition,342,39.728070,34.698830,5.029240,34.502924
14,high_school_microeconomics,267,37.666667,35.084894,2.581773,33.707865
2,astronomy,173,41.219653,37.005780,4.213873,32.947977
32,world_religions,194,11.654639,11.134021,0.520619,31.958763
29,security_studies,211,103.890995,117.733017,-13.842022,31.279621
8,high_school_biology,341,35.331378,32.862170,2.469208,31.085044
23,medical_genetics,116,23.913793,20.807471,3.106322,31.034483
16,high_school_statistics,243,39.609053,37.193416,2.415638,30.864198


### Xuất các sample nghi ngờ answer-length bias

In [8]:
ANSWER_LENGTH_BIAS_CSV = OUT_DIR / "wiki_mcq_answer_length_bias_suspects.csv"

df_len_suspects = df_len[
    (df_len["longest_word_is_correct"] == True)
    & (df_len["correct_minus_avg_distractor_word_len"] >= 5)
].copy()

df_len_suspects = df_len_suspects.sort_values(
    "correct_minus_avg_distractor_word_len",
    ascending=False
)

df_len_suspects.to_csv(ANSWER_LENGTH_BIAS_CSV, index=False)

print("Suspects:", len(df_len_suspects))
print("Wrote:", ANSWER_LENGTH_BIAS_CSV)

display(df_len_suspects[[
    "subject",
    "source_id",
    "answer",
    "correct_word_len",
    "avg_distractor_word_len",
    "correct_minus_avg_distractor_word_len",
    "question",
    "correct_text",
]].head(30))

Suspects: 1313
Wrote: ../seed_exports/wiki_mcq_answer_length_bias_suspects.csv


,subject,source_id,answer,correct_word_len,avg_distractor_word_len,correct_minus_avg_distractor_word_len,question,correct_text
1602,high_school_statistics,high_school_statistics/test/7,D,190,57.000000,133.000000,Chiều cao trung bình của đàn ông trưởng thành ...,"Người phụ nữ, vì chiều cao của cô ấy cao hơn t..."
4327,security_studies,security_studies/test/130,B,240,111.666667,128.333333,Những phê bình thường được đưa ra về HM bao gồ...,HM quá thường cáo buộc chủ nghĩa tư bản hành đ...
1276,security_studies,security_studies/test/19,D,399,276.666667,122.333333,Làm thế nào sự khác biệt sinh học ảnh hưởng đế...,Vai trò giới tính là một khái niệm xã hội; nhữ...
4092,logical_fallacies,logical_fallacies/test/148,B,191,70.666667,120.333333,Mô tả ra tình huống lôi kép?,Nói rằng người tranh luận đối lập đã đưa ra mộ...
1740,security_studies,security_studies/test/67,D,297,179.666667,117.333333,Các phương pháp truyền thống cố gắng giải thíc...,Các phương pháp truyền thống quan tâm đến chủ ...
627,computer_security,computer_security/test/72,C,171,57.666667,113.333333,Khác nhau giữa rò rỉ trực tiếp và kênh bên là gì?,Một sự rò rỉ trực tiếp đến từ cơ chế tương tác...
689,security_studies,security_studies/test/1,B,239,128.000000,111.000000,Liệu thay đổi môi trường có thể được hòa hợp v...,Thay đổi môi trường có thể gây suy yếu an ninh...
2903,security_studies,security_studies/test/93,D,213,109.333333,103.666667,Thêm những triển lãm vũ khí quốc tế và triển l...,Các triển lãm vũ khí và triển lãm vũ khí thườn...
2031,security_studies,security_studies/test/158,D,154,52.000000,102.000000,Phương pháp nào sau đây là hiệu quả để đảm bảo...,Chấp nhận rằng không thể đảm bảo an toàn tuyệt...
7163,high_school_chemistry,high_school_chemistry/val/21,D,174,73.333333,100.666667,Tại sao giấm (một dung dịch loãng của axit axe...,Quá trình hấp thụ nhiệt lớn bất lợi khi tách c...


### N-gram & lexical diversity

In [ ]:
# from pathlib import Path
# import urllib.request

# STOPWORDS_DIR = Path("resources/stopwords")
# STOPWORDS_DIR.mkdir(parents=True, exist_ok=True)

# VI_STOPWORDS_PATH = STOPWORDS_DIR / "vietnamese-stopwords.txt"

# VI_STOPWORDS_URL = (
#     "https://raw.githubusercontent.com/stopwords/vietnamese-stopwords/master/"
#     "vietnamese-stopwords.txt"
# )

# def download_file_if_missing(url, path):
#     if path.exists() and path.stat().st_size > 0:
#         print(f"Stopwords file already exists: {path}")
#         return

#     print(f"Downloading stopwords from: {url}")
#     urllib.request.urlretrieve(url, path)
#     print(f"Saved to: {path}")


# def load_vi_stopwords(path):
#     stopwords = set()

#     with open(path, "r", encoding="utf-8") as f:
#         for line in f:
#             word = line.strip().lower()
#             if not word:
#                 continue
#             if word.startswith("#"):
#                 continue
#             stopwords.add(word)

#     return stopwords


# download_file_if_missing(VI_STOPWORDS_URL, VI_STOPWORDS_PATH)

# VI_STOPWORDS = load_vi_stopwords(VI_STOPWORDS_PATH)

# print("Loaded Vietnamese stopwords:", len(VI_STOPWORDS))
# print("Sample:", sorted(list(VI_STOPWORDS))[:30])

Saved to: resources/stopwords/vietnamese-stopwords.txt
Loaded Vietnamese stopwords: 1942
Sample: ['a ha', 'a lô', 'ai', 'ai ai', 'ai nấy', 'ai đó', 'alô', 'amen', 'anh', 'anh ấy', 'ba', 'ba ba', 'ba bản', 'ba cùng', 'ba họ', 'ba ngày', 'ba ngôi', 'ba tăng', 'bao giờ', 'bao lâu', 'bao nhiêu', 'bao nả', 'bay biến', 'biết', 'biết bao', 'biết bao nhiêu', 'biết chắc', 'biết chừng nào', 'biết mình', 'biết mấy']


In [10]:
VI_STOPWORDS = {
    "là", "của", "và", "các", "một", "những", "cho", "trong", "với",
    "được", "này", "đó", "nào", "sau", "đây", "về", "khi", "có", "không",
    "từ", "theo", "để", "ở", "bởi", "như", "thì", "ra", "vào", "trên",
    "dưới", "nên", "rằng", "hay", "hoặc", "đã", "sẽ", "bị", "làm",
    "đến", "nhiều", "hơn", "nhất", "nếu", "vì", "do", "mà", "nó",
    "anh", "chị", "người", "câu", "hỏi",
}

def tokenize_for_ngram(text):
    text = clean_text(text.lower())
    tokens = re.findall(r"[0-9a-zA-ZÀ-ỹ]+", text)
    tokens = [t for t in tokens if t not in VI_STOPWORDS and len(t) > 1]
    return tokens


def get_ngrams(tokens, n):
    return [
        tuple(tokens[i:i+n])
        for i in range(len(tokens) - n + 1)
    ]

In [11]:
question_rows = []

for i, item in enumerate(cleaned):
    parsed = parse_user_content(item["messages"][0]["content"])
    if parsed is None:
        continue

    q = parsed["question"]

    question_rows.append({
        "row_idx": i,
        "source_id": item["metadata"].get("source_id"),
        "subject": item["metadata"].get("subject"),
        "question": q,
        "question_char_len": len(q),
        "question_word_len": vietnamese_word_count(q),
    })

df_q = pd.DataFrame(question_rows)

print("Question rows:", len(df_q))
display(df_q.head())

Question rows: 7946


,row_idx,source_id,subject,question,question_char_len,question_word_len
0,0,sociology/test/49,sociology,Nhóm Quốc gia Islam kêu gọi đến:,32,25
1,1,miscellaneous/test/535,miscellaneous,Phần nào của phổ điện từ có bước sóng ngắn nhất?,48,37
2,2,human_aging/test/173,human_aging,Các ông bà có cháu đang theo học đại học cho b...,121,94
3,3,astronomy/test/11,astronomy,Làm thế nào Eratosthenes ước tính kích thước c...,88,71
4,4,high_school_mathematics/test/166,high_school_mathematics,Tìm $(2^{20} + 2^{20} + 2^{20} +2^{21}) \div 2...,53,21


In [12]:
bigram_counter = Counter()
trigram_counter = Counter()

for q in df_q["question"]:
    tokens = tokenize_for_ngram(q)
    bigram_counter.update(get_ngrams(tokens, 2))
    trigram_counter.update(get_ngrams(tokens, 3))

df_bigrams = pd.DataFrame([
    {"ngram": " ".join(k), "count": v}
    for k, v in bigram_counter.most_common(50)
])

df_trigrams = pd.DataFrame([
    {"ngram": " ".join(k), "count": v}
    for k, v in trigram_counter.most_common(50)
])

print("Top bigrams:")
display(df_bigrams.head(20))

print("Top trigrams:")
display(df_trigrams.head(20))

Top bigrams:


,ngram,count
0,bao nhiêu,592
1,liên quan,563
2,sử dụng,551
3,chúng ta,486
4,tất cả,427
5,thông tin,423
6,điều gì,333
7,quan thông,297
8,giá trị,291
9,xã hội,275


Top trigrams:


,ngram,count
0,liên quan thông,297
1,quan thông tin,297
2,mối quan hệ,86
3,thể sử dụng,65
4,độ lệch chuẩn,64
5,sự phát triển,63
6,chúng ta thể,63
7,nền kinh tế,62
8,mô tả tốt,51
9,sự khác biệt,47


In [13]:
TEMPLATE_PATTERNS = {
    "which_of_the_following_vi": r"điều nào sau đây|phát biểu nào sau đây|cái nào sau đây|trong số.*sau đây",
    "true_about": r"đúng về|chính xác về|phù hợp nhất",
    "not_true_except": r"không đúng|ngoại trừ|ít có khả năng nhất",
    "best_explains": r"giải thích tốt nhất|mô tả tốt nhất|phù hợp nhất",
}

template_rows = []

for name, pattern in TEMPLATE_PATTERNS.items():
    mask = df_q["question"].str.lower().str.contains(pattern, regex=True, na=False)
    template_rows.append({
        "template": name,
        "count": int(mask.sum()),
        "percent": round(float(mask.mean() * 100), 2),
    })

df_templates = pd.DataFrame(template_rows).sort_values("count", ascending=False)
display(df_templates)

,template,count,percent
0,which_of_the_following_vi,160,2.01
1,true_about,140,1.76
3,best_explains,110,1.38
2,not_true_except,53,0.67


In [14]:
all_tokens = []

for q in df_q["question"]:
    all_tokens.extend(tokenize_for_ngram(q))

unique_tokens = set(all_tokens)

lexical_diversity = {
    "total_tokens": int(len(all_tokens)),
    "unique_tokens": int(len(unique_tokens)),
    "type_token_ratio": round(len(unique_tokens) / max(1, len(all_tokens)), 4),
}

lexical_diversity

{'total_tokens': 187477, 'unique_tokens': 7191, 'type_token_ratio': 0.0384}

In [15]:
NGRAM_BIGRAM_CSV = OUT_DIR / "wiki_mcq_top_bigrams.csv"
NGRAM_TRIGRAM_CSV = OUT_DIR / "wiki_mcq_top_trigrams.csv"
TEMPLATE_REPORT_CSV = OUT_DIR / "wiki_mcq_template_patterns.csv"

df_bigrams.to_csv(NGRAM_BIGRAM_CSV, index=False)
df_trigrams.to_csv(NGRAM_TRIGRAM_CSV, index=False)
df_templates.to_csv(TEMPLATE_REPORT_CSV, index=False)

print("Wrote:", NGRAM_BIGRAM_CSV)
print("Wrote:", NGRAM_TRIGRAM_CSV)
print("Wrote:", TEMPLATE_REPORT_CSV)

Wrote: ../seed_exports/wiki_mcq_top_bigrams.csv
Wrote: ../seed_exports/wiki_mcq_top_trigrams.csv
Wrote: ../seed_exports/wiki_mcq_template_patterns.csv


### Math / Markdown / LaTeX format check

In [16]:
MATH_FORMAT_SUBJECTS = {
    "abstract_algebra",
    "elementary_mathematics",
    "high_school_mathematics",
    "high_school_statistics",
    "conceptual_physics",
    "high_school_physics",
    "high_school_chemistry",
    "astronomy",
}

In [17]:
FORMAT_SUSPICIOUS_PATTERNS = { #  định nghĩa các ký tự thường xuất hiện trong toán học
    "raw_latex_frac": r"\\frac",
    "raw_latex_sqrt": r"\\sqrt",
    "raw_latex_begin": r"\\begin",
    "raw_latex_end": r"\\end",
    "raw_newline_escape": r"\\n",
    "unbalanced_dollar": r"\$",
    "double_backslash": r"\\\\",
    "html_entity": r"&[a-zA-Z]+;",
    "markdown_table_pipe": r"\|",
    "multiple_underscores": r"__+",
    "caret_power": r"\^",
    "subscript_underscore": r"_[A-Za-z0-9]",
    "broken_placeholder": r"\{\s*\}|\[\s*\]",
}

def detect_format_flags(text):
    flags = []
    for name, pattern in FORMAT_SUSPICIOUS_PATTERNS.items():
        if re.search(pattern, text):
            flags.append(name)
    return flags


def count_unbalanced_char(text, char):
    return text.count(char) % 2 != 0

In [18]:
format_issue_rows = []

for i, item in enumerate(cleaned):
    subject = item["metadata"].get("subject")
    if subject not in MATH_FORMAT_SUBJECTS:
        continue

    parsed = parse_user_content(item["messages"][0]["content"])
    if parsed is None:
        continue

    combined_text = "\n".join([
        parsed["question"],
        parsed["choices"]["A"],
        parsed["choices"]["B"],
        parsed["choices"]["C"],
        parsed["choices"]["D"],
    ])

    flags = detect_format_flags(combined_text)

    # check riêng dollar math delimiter
    if count_unbalanced_char(combined_text, "$"):
        flags.append("unbalanced_dollar_count")

    if flags:
        format_issue_rows.append({
            "row_idx": i,
            "source_id": item["metadata"].get("source_id"),
            "subject": subject,
            "answer": get_answer(item),
            "flags": ",".join(sorted(set(flags))),
            "text": combined_text,
        })

df_format_issues = pd.DataFrame(format_issue_rows)

print("Format issue rows:", len(df_format_issues))

if len(df_format_issues) > 0:
    display(df_format_issues.head(30))

Format issue rows: 399


,row_idx,source_id,subject,answer,flags,text
0,4,high_school_mathematics/test/166,high_school_mathematics,B,"caret_power,unbalanced_dollar",Tìm $(2^{20} + 2^{20} + 2^{20} +2^{21}) \div 2...
1,40,high_school_statistics/test/5,high_school_statistics,D,unbalanced_dollar,Hiệu trưởng của một trường đang quan tâm đến v...
2,79,high_school_mathematics/test/160,high_school_mathematics,A,"raw_latex_frac,unbalanced_dollar",Biết rằng một hình chữ nhật có chiều dài $3x$ ...
3,85,abstract_algebra/test/37,abstract_algebra,B,caret_power,Phân định xem đa thức trong Z[x] có thoả Eisen...
4,97,high_school_mathematics/test/154,high_school_mathematics,A,"raw_latex_frac,unbalanced_dollar",Biểu diễn $0.1\overline{7}$ dưới dạng phân số ...
5,107,abstract_algebra/test/62,abstract_algebra,C,markdown_table_pipe,Tuyên bố 1 | Mỗi nhóm phần bất kỳ của mỗi vòng...
6,121,high_school_mathematics/test/200,high_school_mathematics,C,unbalanced_dollar,Grady đi xe đạp nhanh hơn em trai Noah $60\%$....
7,126,high_school_mathematics/test/119,high_school_mathematics,B,"caret_power,unbalanced_dollar","Giả sử $ a, b $ và $ c $ là các số nguyên dươn..."
8,129,high_school_mathematics/test/172,high_school_mathematics,B,"caret_power,unbalanced_dollar","Dãy Fibonacci là dãy 1, 1, 2, 3, 5, $\ldots$ t..."
9,134,abstract_algebra/test/50,abstract_algebra,D,markdown_table_pipe,Phiên bản 1 | Mọi nhóm giải được đều có bội số...


### Format issue summary

In [19]:
format_flag_counter = Counter()

for flags in df_format_issues["flags"] if len(df_format_issues) else []:
    for flag in flags.split(","):
        format_flag_counter[flag] += 1

df_format_flag_summary = pd.DataFrame([
    {"flag": k, "count": v}
    for k, v in format_flag_counter.most_common()
])

display(df_format_flag_summary)

,flag,count
0,unbalanced_dollar,188
1,caret_power,165
2,markdown_table_pipe,66
3,raw_latex_frac,64
4,subscript_underscore,42
5,unbalanced_dollar_count,23
6,raw_latex_sqrt,19
7,multiple_underscores,4


In [20]:
FORMAT_ISSUES_CSV = OUT_DIR / "wiki_mcq_math_format_issues.csv"

df_format_issues.to_csv(FORMAT_ISSUES_CSV, index=False)

print("Wrote:", FORMAT_ISSUES_CSV)
print("Rows:", len(df_format_issues))

Wrote: ../seed_exports/wiki_mcq_math_format_issues.csv
Rows: 399


### Final Micro EDA Report JSON

In [21]:
MICRO_EDA_REPORT_JSON = OUT_DIR / "wiki_mcq_seed_clean_micro_eda_report.json"

micro_eda_report = {
    "input_clean_rows": int(len(cleaned)),

    "answer_length_bias": summary_len,

    "answer_length_bias_by_subject_top20": [
        {
            "subject": str(row["subject"]),
            "rows": int(row["rows"]),
            "avg_correct_word_len": round(float(row["avg_correct_word_len"]), 3),
            "avg_distractor_word_len": round(float(row["avg_distractor_word_len"]), 3),
            "avg_word_gap": round(float(row["avg_word_gap"]), 3),
            "longest_word_correct_rate": round(float(row["longest_word_correct_rate"]), 2),
        }
        for _, row in df_len_by_subject.head(20).iterrows()
    ],

    "lexical_diversity": lexical_diversity,

    "top_bigrams": [
        {"ngram": str(row["ngram"]), "count": int(row["count"])}
        for _, row in df_bigrams.head(20).iterrows()
    ],

    "top_trigrams": [
        {"ngram": str(row["ngram"]), "count": int(row["count"])}
        for _, row in df_trigrams.head(20).iterrows()
    ],

    "template_patterns": [
        {
            "template": str(row["template"]),
            "count": int(row["count"]),
            "percent": round(float(row["percent"]), 2),
        }
        for _, row in df_templates.iterrows()
    ],

    "math_format_issues": {
        "rows": int(len(df_format_issues)),
        "flag_counts": {
            str(k): int(v)
            for k, v in format_flag_counter.items()
        },
        "csv": str(FORMAT_ISSUES_CSV),
    },

    "artifacts": {
        "answer_length_bias_suspects_csv": str(ANSWER_LENGTH_BIAS_CSV),
        "top_bigrams_csv": str(NGRAM_BIGRAM_CSV),
        "top_trigrams_csv": str(NGRAM_TRIGRAM_CSV),
        "template_patterns_csv": str(TEMPLATE_REPORT_CSV),
        "math_format_issues_csv": str(FORMAT_ISSUES_CSV),
    }
}

with open(MICRO_EDA_REPORT_JSON, "w", encoding="utf-8") as f:
    json.dump(micro_eda_report, f, ensure_ascii=False, indent=2)

In [22]:
print(json.dumps(micro_eda_report, ensure_ascii=False, indent=2))

{
  "input_clean_rows": 7946,
  "answer_length_bias": {
    "rows": 7946,
    "avg_correct_word_len": 27.752,
    "avg_distractor_word_len": 26.959,
    "avg_word_len_gap_correct_minus_distractor": 0.793,
    "avg_correct_char_len": 35.596,
    "avg_distractor_char_len": 34.507,
    "avg_char_len_gap_correct_minus_distractor": 1.089,
    "longest_word_choice_is_correct_rate": 27.38,
    "longest_char_choice_is_correct_rate": 27.57
  },
  "answer_length_bias_by_subject_top20": [
    {
      "subject": "management",
      "rows": 119,
      "avg_correct_word_len": 20.857,
      "avg_distractor_word_len": 18.756,
      "avg_word_gap": 2.101,
      "longest_word_correct_rate": 35.29
    },
    {
      "subject": "logical_fallacies",
      "rows": 185,
      "avg_correct_word_len": 37.627,
      "avg_distractor_word_len": 34.004,
      "avg_word_gap": 3.623,
      "longest_word_correct_rate": 34.59
    },
    {
      "subject": "nutrition",
      "rows": 342,
      "avg_correct_word_len": 3